In [1]:
import os
import json
import data
import random
import sklearn
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.utils import shuffle, check_random_state

import config
from utils import apply_symmetric_noise

random.seed(config.SEED)
np.random.seed(config.SEED)
np.random.default_rng(config.SEED)
check_random_state(config.SEED)

import warnings

warnings.filterwarnings("ignore")

In [2]:
DATA_SIZE = 100000
MAX_ITER = 10000
COLLECTION_NAME = "Swimmer"

In [3]:
if COLLECTION_NAME == "UTK":
    array = shuffle(data.load_utk())[:DATA_SIZE]
elif COLLECTION_NAME == "Olivetti":
    array = shuffle(data.load_olivetti())[:DATA_SIZE]
elif COLLECTION_NAME == "Swimmer":
    array = shuffle(data.load_swimmer())[:DATA_SIZE]
else:
    raise ValueError

In [4]:
rank_list = [4, 8, 12, 16, 20, 24, 28, 32, 36, 40]
results_list = list()
for rank in tqdm(rank_list):
    model = NMF(n_components=rank, max_iter=MAX_ITER, init="random", random_state=config.SEED)
    W = model.fit_transform(array)
    H = model.components_
    array_recon = W @ H

    frac_pi_array = array.sum(axis=0) / array_recon.sum(axis=0) - 1
    abs_frac_pi_mean = abs(frac_pi_array.mean()).item()
    frac_pi_std = np.std(frac_pi_array, ddof=1).item()

    frac_i_array = array.sum(axis=1) / array_recon.sum(axis=1) - 1
    abs_frac_i_mean = abs(frac_i_array.mean()).item()
    frac_i_std = np.std(frac_i_array, ddof=1).item()

    log_row = [COLLECTION_NAME, rank, abs_frac_pi_mean, frac_pi_std, abs_frac_i_mean, frac_i_std]
    results_list.append(log_row)

100%|███████████████████████████████████████████| 10/10 [00:00<00:00, 63.86it/s]


In [5]:
df = pd.DataFrame(
    results_list,
    columns = ["Collection", "Rank", "Last mean by pi", "Last var by pi", "Last mean by i", "Last var by i"]
)
df

,Collection,Rank,Last mean by pi,Last var by pi,Last mean by i,Last var by i
0,Swimmer,4,NaN,NaN,0.016411,0.012363
1,Swimmer,8,NaN,NaN,0.039041,0.025745
2,Swimmer,12,NaN,NaN,0.051693,0.020279
3,Swimmer,16,NaN,NaN,0.025940,0.006065
4,Swimmer,20,NaN,NaN,0.000049,0.000087
5,Swimmer,24,NaN,NaN,0.000093,0.000217
6,Swimmer,28,NaN,NaN,0.000014,0.000036
7,Swimmer,32,NaN,NaN,0.000010,0.000021
8,Swimmer,36,NaN,NaN,0.000126,0.000263
9,Swimmer,40,NaN,NaN,0.000044,0.000142


In [6]:
df.to_excel(f"{COLLECTION_NAME}_check_normalizations.xlsx", index=False)

In [10]:
df1 = pd.read_excel("UTK_check_normalizations.xlsx")
df2 = pd.read_excel("Olivetti_check_normalizations.xlsx")
df3 = pd.read_excel("Swimmer_check_normalizations.xlsx")

df = pd.concat([df1, df2, df3]).reset_index(drop=True)
df.to_excel("check_normalizations.xlsx", index=False)